# Train `ASCIIVAE` on Google Colab

Reads the ASCII shards produced by `notebooks/collect_gvgai_ascii_colab.ipynb` and trains `world_model.model.net.ascii_vae.ASCIIVAE` via `world_model/train_ascii_vae.py`. Both notebooks share the same Drive layout:

```
MyDrive/DecoupliWo/
  data/transitions/{train,test}/<game>/<variant>/shard_XXXXX/  # written by collection
  checkpoints/ascii_vae/vae.pt                                  # written here
  runs/ascii_vae/<timestamp>/                                   # TensorBoard here
```

## Runtime

1. `Runtime` → `Change runtime type` → **GPU** → pick **A100** or **H100** (Pro / Pro+ only). T4 also works; the model is tiny.
2. Enable **High-RAM** if offered.

## Prerequisite

Run `collect_gvgai_ascii_colab.ipynb` at least once so that `MyDrive/DecoupliWo/data/transitions/{train,test}/...` contains shards. You can re-run that notebook later to grow the dataset; this notebook always picks up whatever is currently on Drive.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Canonical paths (shared with the collection notebook)

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/wessimpson/DecoupliWo.git'
REPO_BRANCH = 'gameNGen-world_model-v2'
REPO_DIR = Path('/content/DecoupliWo')

DRIVE_ROOT = Path('/content/drive/MyDrive/DecoupliWo')
DRIVE_DATA_DIR = DRIVE_ROOT / 'data' / 'transitions'
DRIVE_CKPT_DIR = DRIVE_ROOT / 'checkpoints' / 'ascii_vae'
DRIVE_RUNS_DIR = DRIVE_ROOT / 'runs' / 'ascii_vae'

LOCAL_DATA_DIR = REPO_DIR / 'data' / 'transitions'

DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)

assert DRIVE_DATA_DIR.is_dir() and any(DRIVE_DATA_DIR.rglob('shard_*')), (
    f'No shards under {DRIVE_DATA_DIR}. Run collect_gvgai_ascii_colab.ipynb first.'
)
print('OK drive data dir:', DRIVE_DATA_DIR)
print('OK ckpt dir     :', DRIVE_CKPT_DIR)
print('OK runs dir     :', DRIVE_RUNS_DIR)

## 4. Clone the repo

In [ ]:
import subprocess

if REPO_DIR.is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(
        ['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)
print('cwd =', os.getcwd())
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)

## 5. Install minimal dependencies

The full `requirements.txt` drags in Atari / OCAtari / stable-baselines3 / diffusers / etc. None of those are needed to train `ASCIIVAE`. We only need `numpy`, `tensorboard`, `tqdm`, and PyTorch (pre-installed on Colab GPU runtimes).

In [ ]:
!pip install --quiet 'numpy<2' tensorboard tqdm

## 6. Sync shards Drive → local SSD

Reading `.npy` files via `mmap_mode='r'` over the Drive FUSE mount is extremely slow (per-page network round-trips). One-time `rsync` to the Colab VM's local NVMe takes a few minutes and then training reads are fast. `rsync` is incremental: subsequent runs only copy new/changed shards.

In [ ]:
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
!rsync -a --info=progress2 "{DRIVE_DATA_DIR}/" "{LOCAL_DATA_DIR}/"
print('train shards:', len(list((LOCAL_DATA_DIR / 'train').rglob('shard_*'))))
print('test  shards:', len(list((LOCAL_DATA_DIR / 'test').rglob('shard_*'))))

## 7. Sanity check: one forward pass

In [ ]:
import sys
import torch

sys.path.insert(0, str(REPO_DIR))

from world_model.ascii.constants import CANVAS_H, CANVAS_W, VOCAB_SIZE
from world_model.model.net.ascii_vae import ASCIIVAE
from world_model.train_ascii_vae import AllAsciiFramesDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ds = AllAsciiFramesDataset(LOCAL_DATA_DIR / 'train', CANVAS_H, CANVAS_W)
print(f'train frames = {len(ds):,}   canvas = {CANVAS_H}x{CANVAS_W}   vocab = {VOCAB_SIZE}')

vae = ASCIIVAE().to(device)
ids = torch.stack([ds[i] for i in range(4)]).to(device)
logits, mu, logvar = vae(ids)
print('ids    ', tuple(ids.shape), ids.dtype)
print('logits ', tuple(logits.shape))
print('mu     ', tuple(mu.shape))
print('params ', sum(p.numel() for p in vae.parameters()))

## 8. Launch TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{DRIVE_RUNS_DIR}"

## 9. Train

Flag notes:

- `--output_dir` and `--log_dir` point at Drive so checkpoints and TensorBoard logs survive VM termination.
- Tune `--batch_size` / `--num_workers` for your GPU. H100/A100: 256 / 4 is comfortable. T4: drop to 128 / 2.
- **Match `--epochs` × steps-per-epoch to `--max_train_steps`** so the shorter of the two doesn't truncate training. With few thousand frames you want `--epochs` large (e.g. 300) and `--max_train_steps` as the real cap.

### Online mode (train while collection runs in another Colab tab)

Two new flags make the trainer stream from Drive:

- `--sync_from <drive path>`: before each refresh, `rsync` this path into `--train_dir`. Point it at `DRIVE_DATA_DIR/train`.
- `--refresh_every <N>`: every N optimizer steps, `_maybe_sync` then rescan `--train_dir` for new `shard_*` directories. Dataset grows in place; the DataLoader is rebuilt so workers see the new shards. `0` disables (sequential mode).

Workflow:

1. Open a **second Colab tab** with `collect_gvgai_ascii_colab.ipynb` on a **CPU** runtime. Start it collecting.
2. As soon as any shards appear under `MyDrive/DecoupliWo/data/transitions/train/`, start this notebook on a **GPU** runtime. Cell 6 grabs whatever is currently there; the trainer below keeps pulling new shards every `--refresh_every` steps.
3. Leave both tabs running. Training stops at `--max_train_steps`; collection stops at its own frame budget. If one finishes first, the other keeps going without needing intervention.

Requires Colab Pro or Pro+ to run two runtimes concurrently on the same account. Free tier usually only allows one active runtime and will evict the older session.

In [ ]:
!python -m world_model.train_ascii_vae \
  --train_dir "{LOCAL_DATA_DIR}/train" \
  --val_dir   "{LOCAL_DATA_DIR}/test"  \
  --output_dir "{DRIVE_CKPT_DIR}" \
  --log_dir    "{DRIVE_RUNS_DIR}" \
  --sync_from  "{DRIVE_DATA_DIR}/train" \
  --refresh_every 500 \
  --batch_size 256 \
  --num_workers 4 \
  --epochs 1000000 \
  --max_train_steps 20000 \
  --save_every 2000 \
  --validation_every 500 \
  --log_every 50

## 10. Verify saved checkpoint

In [ ]:
from world_model.model.net.ascii_vae import load_ascii_vae

ckpt = DRIVE_CKPT_DIR / 'vae.pt'
assert ckpt.is_file(), f'no checkpoint at {ckpt}'

vae = load_ascii_vae(ckpt).to(device)
vae.train(False)
ids = torch.stack([ds[i] for i in range(4)]).to(device)
logits, _, _ = vae(ids)
pred = ASCIIVAE.logits_to_ids(logits)
acc = (pred == ids).float().mean().item()
print(f'checkpoint    : {ckpt}  ({ckpt.stat().st_size / 1e6:.2f} MB)')
print(f'scaling_factor: {vae.scaling_factor:.4f}')
print(f'per-cell acc  : {acc:.4f} on 4 sample frames')